# CONNECT TO MYSQL VIA SQLALCHEMY

In [1]:
import pandas as pd
from sqlalchemy import create_engine

from sqlalchemy.engine import URL

url_object = URL.create(
    "mysql+pymysql",
    username="root",
    password="saraswathi@2026",  
    host="localhost",
    database="inventory_management",
)

engine = create_engine(url_object)

print("Engine created. Ready to pull views.")

Engine created. Ready to pull views.


# PULL THE KEY VIEWS FOR STATS

In [2]:
# Pulling the key views for our planned stats
df_lead_source = pd.read_sql("SELECT * FROM lead_source_summary", engine)
df_region_division = pd.read_sql("SELECT * FROM region_division_summary", engine)
df_order_delays = pd.read_sql("SELECT * FROM order_delay_reason_metrics", engine)
df_lead_conversion = pd.read_sql("SELECT * FROM lead_conversion", engine)
df_lead_velocity=pd.read_sql("SELECT * FROM lead_velocity", engine)
df_lead_lifetracker=pd.read_sql("SELECT * FROM lead_lifetracker", engine)

print("Views loaded into DataFrames")

Views loaded into DataFrames


# Chi-Square Test: Is Success tied to Region ("REGION_DIVISION VIEW")

In [3]:
import scipy.stats as stats

# 1. Create a contingency table (Region vs Won/Lost)
# We use the counts directly from your view
contingency_table = df_region_division.set_index('region')[['won_leads', 'lost_leads']]

# 2. Run the Chi-Square test
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print("--- Chi-Square Test: Region vs. Conversion ---")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_val:.4f}")

if p_val < 0.05:
    print("Result: Statistically Significant. Location IS a predictor of success.")
else:
    print("Result: Not Significant. Regional differences are likely due to random chance.")

--- Chi-Square Test: Region vs. Conversion ---
Chi-Square Statistic: 44.2865
P-Value: 0.0728
Result: Not Significant. Regional differences are likely due to random chance.


# Calculate Bayesian Probability for each source
# P(Win | Source) = Won Count / Total Leads ("LEAD SOURCE SUMMARY VIEW")

In [4]:
# Bayesian Probability: Predictive Likelihood

df_lead_source['conversion_prob'] = df_lead_source['won_count'] / df_lead_source['total_leads']

# Sort to see your "High Probability" channels
predictive_power = df_lead_source[['name', 'conversion_prob']].sort_values(by='conversion_prob', ascending=False)

print("--- Bayesian Conversion Likelihood ---")
print(predictive_power)
predictive_power.to_csv('source_predictive_likelihood.csv')

--- Bayesian Conversion Likelihood ---
             name  conversion_prob
0       Cold Call         0.542857
1        Referral         0.400000
2      Event/Expo         0.349206
3           Visit         0.346154
4   Advertisement         0.315789
5  Email Campaign         0.302326
6  Tender/Enquiry         0.291667
7         Website         0.275000


# One-Way ANOVA: Operational Bottlenecks ("RAW TABLES : ORDERS AND REASON_LOOKUP")

In [5]:
import pandas as pd

# Joining the tables directly in the SQL query
sql_query = """
SELECT 
    r.name AS delay_reason, 
    DATEDIFF(o.order_out_date, o.order_in_date) AS delay_days
FROM orders o
JOIN reason_lookup r ON o.reason_id = r.reason_id
"""

# Load the raw observations into a DataFrame
df_raw_delays = pd.read_sql(sql_query, engine)

print(f"Total Observations: {len(df_raw_delays)}")
print(df_raw_delays.groupby('delay_reason').size()) # Check sample size per group

Total Observations: 127
delay_reason
Customer Hold         28
Damage                19
Installation delay    22
Logistics Delay       19
Payment Pending       19
Stock Shortage        20
dtype: int64


In [6]:
import scipy.stats as stats

# Grouping by reason
groups = [group['delay_days'].values for name, group in df_raw_delays.groupby('delay_reason')]

# Running ANOVA
f_stat, p_val = stats.f_oneway(*groups)

print("--- One-Way ANOVA Result ---")
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_val:.4f}")

if p_val < 0.05:
    print("INSIGHT: The difference in delay times is statistically significant.")
else:
    print("INSIGHT: No significant difference; delays are consistent across reasons.")

--- One-Way ANOVA Result ---
F-Statistic: 108.4463
P-Value: 0.0000
INSIGHT: The difference in delay times is statistically significant.


In [8]:
# Create a list to hold all your test results
test_results = []

# After running your ANOVA:
result_text = "Significant" if p_val < 0.05 else "Not Significant"

test_results.append({
    'Test_Category': 'Operations',
    'Test_Name': 'ANOVA: Delay Reason vs Time',
    'Metric': 'F-Statistic',
    'Value': round(f_stat, 4),
    'P_Value': round(p_val, 4),
    'Outcome': result_text
})

# Convert to DataFrame and Save
df_stats_summary = pd.DataFrame(test_results)
df_stats_summary.to_csv('ms_operational_bottleneck_insights.csv', index=False)

# Identify the highest risk based on variance

In [9]:
unreliable_process = df_order_delays.loc[df_order_delays['var_delay'].idxmax()]

print("--- Operational Risk Alert ---")
print(f"Most Unpredictable Process: {unreliable_process['name']}")
print(f"Variance: {unreliable_process['var_delay']}")

--- Operational Risk Alert ---
Most Unpredictable Process: Payment Pending
Variance: 9.667590027700827


## Above result stored into CSV

In [10]:
import pandas as pd

# 1. Get the most unpredictable process (your current logic)
unreliable_process = df_order_delays.loc[df_order_delays['var_delay'].idxmax()]

# 2. Create a 'Risk Alert' DataFrame for the top offender
df_risk_alert = pd.DataFrame([{
    'Metric': 'Most Unpredictable Process',
    'Process_Name': unreliable_process['name'],
    'Variance_Value': round(unreliable_process['var_delay'], 2),
    'Risk_Level': 'Critical' if unreliable_process['var_delay'] > df_order_delays['var_delay'].mean() * 2 else 'Moderate'
}])

# 3. Save the full variance table (for the Tableau Leaderboard)
df_order_delays.to_csv('operational_variance_analysis.csv', index=False)

# 4. Save the specific high-level alert separately
df_risk_alert.to_csv('high_priority_risks.csv', index=False)

print("Operational Risk Analysis saved to CSV.")

Operational Risk Analysis saved to CSV.


# Chi sq test to see if conversion success is dependent of region. "Region-Division_view"

In [11]:
import scipy.stats as stats

# Take a quick look to ensure 'won_leads' and 'lost_leads' columns are there
print(df_region_division[['region', 'won_leads', 'lost_leads']].head())

        region  won_leads  lost_leads
0        Delhi        4.0         2.0
1  Maharashtra        7.0         3.0
2      Gujarat        6.0         3.0
3  West Bengal        2.0         1.0
4       Others        3.0         1.0


In [12]:
# We set 'region' as the index so the table contains only the numeric counts
contingency_table = df_region_division.set_index('region')[['won_leads', 'lost_leads']]

print("Contingency Table for Chi-Square:")
print(contingency_table)

Contingency Table for Chi-Square:
             won_leads  lost_leads
region                            
Delhi              4.0         2.0
Maharashtra        7.0         3.0
Gujarat            6.0         3.0
West Bengal        2.0         1.0
Others             3.0         1.0
Maharashtra       14.0        16.0
Karnataka          7.0         3.0
Telangana         12.0         7.0
Karnataka          4.0         8.0
Goa                4.0         0.0
Others             4.0         9.0
Gujarat            5.0         3.0
Kerala             1.0         4.0
Gujarat            5.0         9.0
Kerala             5.0         8.0
Telangana          2.0         3.0
Rajasthan          2.0         3.0
Telangana          7.0         1.0
Karnataka          4.0         6.0
Delhi              2.0         1.0
Tamil Nadu         3.0         5.0
Maharashtra        3.0         4.0
Tamil Nadu         3.0         2.0
Goa                1.0         2.0
Rajasthan          3.0         3.0
Delhi              4.

In [16]:
# Run the test
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print("\n--- Chi-Square Test: Region vs. Success ---")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_val:.4f}")
print(f"Degrees of Freedom: {dof}")

if p_val < 0.05:
    print("RESULT: Significant. Conversion success IS dependent on the Region.")
else:
    print("RESULT: Not Significant. Regional differences are likely due to random chance.")


--- Chi-Square Test: Region vs. Success ---
Chi-Square Statistic: 44.2865
P-Value: 0.0728
Degrees of Freedom: 32
RESULT: Not Significant. Regional differences are likely due to random chance.


In [13]:
# Calculate the actual conversion % per region for your dashboard labels
df_region_division['conv_rate'] = (df_region_division['won_leads'] / (df_region_division['won_leads'] + df_region_division['lost_leads'])) * 100

# Export to CSV for Tableau
df_region_division.to_csv('regional_conversion_stats.csv', index=False)

# Chi-Square Test: Division vs. Success

In [14]:
contingency_division = df_region_division.groupby('division')[['won_leads', 'lost_leads']].sum()

print("Contingency Table: Division Performance")
print(contingency_division)
# Run the test of independence
chi2_div, p_val_div, dof_div, expected_div = stats.chi2_contingency(contingency_division)

print("\n--- Chi-Square Test: Division vs. Success ---")
print(f"Chi-Square Statistic: {chi2_div:.4f}")
print(f"P-Value: {p_val_div:.4f}")

if p_val_div < 0.05:
    print("INSIGHT: Success is highly dependent on the Division.")
else:
    print("INSIGHT: No statistical difference between Divisions (Random distribution).")

Contingency Table: Division Performance
                 won_leads  lost_leads
division                              
BatterResearch        31.0        24.0
LifeScience           58.0        86.0
MaterialScience       38.0        27.0

--- Chi-Square Test: Division vs. Success ---
Chi-Square Statistic: 7.8293
P-Value: 0.0199
INSIGHT: Success is highly dependent on the Division.


## Above saved into Csv

In [15]:
test_results = []

# 1. Capture the Division Chi-Square Result
div_outcome = "Dependent (Significant)" if p_val_div < 0.05 else "Independent (Not Significant)"

test_results.append({
    'Test_Category': 'Strategic',
    'Test_Name': 'Chi-Square: Division vs. Success',
    'Metric': 'Chi2-Stat',
    'Value': round(chi2_div, 4),
    'P_Value': round(p_val_div, 4),
    'Outcome': div_outcome
})

# 2. Convert the full list of all tests to a CSV
df_master_stats = pd.DataFrame(test_results)
df_master_stats.to_csv('division_success.csv', index=False)

print("Summary updated and saved to CSV.")

Summary updated and saved to CSV.


# Calculate win rates to see which division is the "star"

In [16]:
df_div_stats = contingency_division.copy()
df_div_stats['total'] = df_div_stats['won_leads'] + df_div_stats['lost_leads']
df_div_stats['win_rate'] = (df_div_stats['won_leads'] / df_div_stats['total']) * 100

# Sort it to make the "Sharp" insight obvious
df_div_stats = df_div_stats.sort_values(by='win_rate', ascending=False)

print("Final Division Stats for Tableau:")
print(df_div_stats)

# Export this for your dashboard
df_div_stats.to_csv('division_significance_results.csv')

Final Division Stats for Tableau:
                 won_leads  lost_leads  total   win_rate
division                                                
MaterialScience       38.0        27.0   65.0  58.461538
BatterResearch        31.0        24.0   55.0  56.363636
LifeScience           58.0        86.0  144.0  40.277778


# Revenue_dependency CHI SQ TEST ("PRODUCT_PROFIT SUMMARY VIEW")

In [17]:
# REgion division

# Pull the revenue/profit summary
df_profit = pd.read_sql("SELECT * FROM product_profit_summary", engine)


In [18]:
# Calculate the median revenue
threshold = df_profit['total_revenue'].median()

# Create the categorical 'Revenue_Class'
df_profit['rev_class'] = df_profit['total_revenue'].apply(
    lambda x: 'High' if x >= threshold else 'Low'
)

print(f"Revenue Threshold set at: {threshold}")

Revenue Threshold set at: 1975000.0


In [19]:
# --- Test 2: Revenue vs. Division ---
contingency_div = pd.crosstab(df_profit['division'], df_profit['rev_class'])
chi2_div, p_div, _, _ = stats.chi2_contingency(contingency_div)

print(f"Division vs Revenue: p-value = {p_div:.4f}")

Division vs Revenue: p-value = 0.1765


In [20]:
# Save for your dashboard
df_profit.to_csv('revenue_dependency_results.csv', index=False)

# BAYSEIAN PROBABILITY:P(High Revenue | Division) "PRODUCT_PROFIT VIEW"

In [21]:
# 1. Calculate the 'Prior': The general probability of any lead being High Revenue
total_leads = len(df_profit)
high_rev_leads = len(df_profit[df_profit['rev_class'] == 'High'])
p_prior_high = high_rev_leads / total_leads

# 2. Calculate the 'Likelihood' per Division
# This is P(High Revenue | Division)
bayesian_scorecard = df_profit.groupby('division').apply(
    lambda x: (x['rev_class'] == 'High').sum() / len(x)
).reset_index()

bayesian_scorecard.columns = ['division', 'prob_high_revenue']

# 3. Sort for the "Sharp" Dashboard insight
bayesian_scorecard = bayesian_scorecard.sort_values(by='prob_high_revenue', ascending=False)

print("--- Bayesian Predictive Scorecard ---")
print(bayesian_scorecard)

--- Bayesian Predictive Scorecard ---
          division  prob_high_revenue
2  MaterialScience           0.727273
1      LifeScience           0.411765
0   BatterResearch           0.333333


C:\Users\Sukruth\AppData\Local\Temp\ipykernel_21724\520842691.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bayesian_scorecard = df_profit.groupby('division').apply(


In [22]:
# Export the probabilities for Tableau tooltips and gauges
bayesian_scorecard.to_csv('division_bayesian_scores.csv', index=False)

# Lead Velocity & Pipeline Conversion Leakage. ("LEAD_CONVERSION VIEW")

In [23]:
# Lead Velocity & Pipeline Conversion Leakage.
import pandas as pd

# Load the data
df_conv = pd.read_sql("SELECT * FROM lead_conversion", engine)

# 1. Calculate 'Outcome Rate' (How many leads actually reached a decision)
df_conv['outcome_count'] = df_conv['won_count'] + df_conv['lost_count']
df_conv['velocity_score'] = (df_conv['outcome_count'] / df_conv['total_leads']) * 100

# 2. Calculate 'Win-to-Loss Ratio' (Efficiency of the decided leads)
df_conv['success_ratio'] = df_conv['won_count'] / df_conv['lost_count']

print(df_conv[['month', 'total_leads', 'velocity_score', 'success_ratio']])
df_conv.to_csv('pipeline_leakage_results.csv', index=False)

      month  total_leads  velocity_score  success_ratio
0   2025-04           50       22.000000       1.200000
1   2025-05           28      125.000000       1.500000
2   2025-06           31       87.096774       0.588235
3   2025-07           31      116.129032       0.800000
4   2025-08           31       70.967742       0.833333
5   2025-09           39       76.923077       1.727273
6   2025-10           30      130.000000       0.857143
7   2025-11           28       57.142857       0.454545
8   2025-12           44       47.727273       1.100000
9   2026-01           15      120.000000       0.636364
10  2026-02           18       38.888889       0.400000
11  2026-03           18       22.222222       1.000000


# Lead Volume Seasonality Test ("LEAD VELOCITY VIEW")

In [24]:
import pandas as pd
import scipy.stats as stats

# 1. Pull the counts from your view
df_velocity = pd.read_sql("SELECT month, lead_count FROM lead_velocity", engine)

# 2. Run the Chi-Square Goodness-of-Fit
# We compare observed counts vs. the mean (expected if it were perfectly even)
observed = df_velocity['lead_count']
expected = [observed.mean()] * len(observed)

chi_stat, p_val = stats.chisquare(f_obs=observed, f_exp=expected)

print(f"--- Lead Volume Seasonality Test ---")
print(f"Chi-Square Statistic: {chi_stat:.4f}")
print(f"P-Value: {p_val:.4f}")

if p_val < 0.05:
    print("INSIGHT: Significant Seasonality. Lead volume is NOT random; certain months peak.")
else:
    print("INSIGHT: Random Distribution. Variation is just noise; volume is stable over time.")

--- Lead Volume Seasonality Test ---
Chi-Square Statistic: 39.6777
P-Value: 0.0000
INSIGHT: Significant Seasonality. Lead volume is NOT random; certain months peak.


In [25]:
test_results = []

# 1. Determine the outcome string
seasonality_outcome = "Seasonal (Significant)" if p_val < 0.05 else "Uniform/Stable (Not Significant)"

# 2. Append the results
test_results.append({
    'Test_Category': 'Marketing',
    'Test_Name': 'Chi-Square: Lead Volume Seasonality',
    'Metric': 'Chi2-Stat',
    'Value': round(chi_stat, 4),
    'P_Value': round(p_val, 4),
    'Outcome': seasonality_outcome
})

# 3. Final Export of the Master Summary
df_master_stats = pd.DataFrame(test_results)
df_master_stats.to_csv('seasonal_lead_significance.csv', index=False)

print("Seasonality test added.")

Seasonality test added.


In [26]:
import pandas as pd

# Joining the main Leads table to its two direct Lookups
sql_path = """
SELECT 
    sl.name AS source_name,
    st.name AS status_name
FROM leads l
JOIN lead_source_lookup sl ON l.source_id = sl.source_id
JOIN lead_status_lookup st ON l.status_id = st.status_id
"""

try:
    # 1. Pull the current state of all leads
    df_leads = pd.read_sql(sql_path, engine)
    
    # 2. Create the Cross-tabulation (Source vs. Status)
    # This counts how many leads from each source are currently in each status
    ct = pd.crosstab(df_leads['source_name'], df_leads['status_name'])
    
    # 3. Calculate the Row-wise Probability
    # (Number of leads in Status X / Total leads from Source Y)
    prob_matrix = ct.div(ct.sum(axis=1), axis=0) * 100
    
    print("--- Lead Source to Current Status Probability (%) ---")
    print(prob_matrix.round(2))

except Exception as e:
    print(f"SQL Error: {e}")
    
prob_matrix.to_csv('source_status_probabilities.csv')

--- Lead Source to Current Status Probability (%) ---
status_name     Cold  Deferred  Demo Scheduled   Hot   Lost  Negotiation  \
source_name                                                                
Advertisement   7.89      5.26            2.63  0.00  34.21         0.00   
Cold Call       5.71      5.71            2.86  0.00  25.71         0.00   
Email Campaign  2.33     13.95            4.65  0.00  34.88         0.00   
Event/Expo      3.17      3.17            3.17  1.59  41.27         1.59   
Referral        2.22     13.33            0.00  0.00  31.11         0.00   
Tender/Enquiry  2.08      2.08            6.25  0.00  41.67         8.33   
Visit           1.92      3.85            3.85  0.00  46.15         3.85   
Website         2.50      5.00            2.50  0.00  40.00         5.00   

status_name     New Lead  On Hold  Qualified  Tendered  Warm    Won  
source_name                                                          
Advertisement       5.26     2.63       2.63 

In [27]:
import pandas as pd

# Load the separate stat files
df_div = pd.read_csv('ms_division_success.csv')
df_ops = pd.read_csv('ms_operational_bottleneck_insights.csv')
df_sea = pd.read_csv('ms_seasonal_lead_significance.csv')

# Merge them into one
df_master_stats = pd.concat([df_div, df_ops, df_sea], ignore_index=True)

# Save as the single Master file
df_master_stats.to_csv('master_statistical_summary.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'ms_division_success.csv'

In [ ]:
import pandas as pd
import numpy as np

# For the Relationship Analysis
from scipy import stats 

# For PCA (The M.Tech "Sharp" Analysis)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# For the Visual Checks in Jupyter
import seaborn as sns
import matplotlib.pyplot as plt

# Spearman correlation matrix for "won_lead-stats" view followed by PCA for the same

In [28]:
# 1. Load the data from your new view
df_won = pd.read_sql("SELECT days_to_win, revenue FROM won_lead_stats", engine)

# 2. Correlation Matrix (Strength: -1 to 1)
# We use 'pearson' for linear and 'spearman' for rank-based relationships
won_corr = df_won.corr(method='pearson')

# 3. Covariance Matrix (Direction and Scale)
won_cov = df_won.cov()

# 4. Save to CSV for Tableau
won_corr.to_csv('ms_won_lead_correlation.csv')
won_cov.to_csv('ms_won_lead_covariance.csv')

print("Matrices created. Use 'ms_won_lead_correlation.csv' for your Tableau Heatmap.")

Matrices created. Use 'ms_won_lead_correlation.csv' for your Tableau Heatmap.


In [29]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

query = "SELECT lead_id, days_to_win, revenue FROM won_lead_stats"
df_won = pd.read_sql(query, engine)
# 1. Scale the data
features = ['days_to_win', 'revenue']
x = df_won[features]
x_scaled = StandardScaler().fit_transform(x)

# 2. Run PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(x_scaled)

# 3. Create a DataFrame for Tableau
pca_df = pd.DataFrame(data = principalComponents, columns = ['PC1', 'PC2'])

# KEY FIX: Map the actual Lead IDs back to the PCA results
# We use .reset_index(drop=True) on df_won temporarily if needed, 
# but simply grabbing the values is the most direct way:
pca_df['lead_id'] = df_won['lead_id'].values 

# 4. Save for Tableau
# Ensure the delimiter matches your other files (Semicolon based on your screenshot)
pca_df.to_csv('ms_pca_results.csv', index=False, sep=';')

print("PCA Complete. Variance captured:", pca.explained_variance_ratio_)

PCA Complete. Variance captured: [0.51044896 0.48955104]


# Friction index (Correlate Delay_days with lost_status)

In [30]:
from scipy import stats
# 1. Load data
df_friction = pd.read_sql("SELECT total_stay_days, is_lost FROM friction_analysis", engine)

# 2. Calculate Point-Biserial Correlation
# This measures how strongly 'staying longer' correlates with 'losing'
correlation, p_value = stats.pointbiserialr(df_friction['is_lost'], df_friction['total_stay_days'])

print(f"Friction Index (Correlation): {correlation:.4f}")
print(f"Significance (P-value): {p_value:.4f}")

Friction Index (Correlation): -0.0354
Significance (P-value): 0.5010


# Stability LeaderBoard "Source stability view"

In [31]:
df_stability = pd.read_sql("SELECT * FROM source_stability", engine)

# Sort by CV (Lower is better/more stable)
df_stability = df_stability.sort_values(by='cv_index')

print("Source Stability Analysis:")
print(df_stability[['source_name', 'avg_revenue', 'cv_index']])

Source Stability Analysis:
      source_name   avg_revenue  cv_index
3  Email Campaign  6.176923e+05  0.573228
1        Referral  7.805556e+05  0.693006
5  Tender/Enquiry  7.671429e+05  0.788913
2      Event/Expo  7.245455e+05  0.936651
0   Advertisement  1.930583e+06  1.214666
4       Cold Call  1.308947e+06  1.545904
6           Visit  1.131222e+06  1.842233
7         Website  1.827273e+06  1.957021
